# Two independently trained Transformer pipelines

This notebook blends two separately trained prediction pipelines:

1. **Raw Transformer:** ordinary Transformer training and raw visit predictions.
2. **Post-processed Transformer:** a second, independently initialized Transformer trained with the same ordinary loss; only after prediction, each patient's trajectory is fitted by a quadratic least-squares curve.

There is **no backpropagation through the quadratic model**. The quadratic step is a pure post-processing operation fitted independently for each patient's predicted visits.

Fold 1 is used only to estimate nonnegative blending coefficients that sum to one. The coefficients are frozen before folds 2–5 are trained and evaluated. Both pipelines are evaluated for base seeds 42, 43 and 44, and predictions are averaged across seeds before blending.

The primary metric is visit-weighted OOF MSE on folds 2–5, with patient-cluster uncertainty. The final-training cell is disabled by default (`RUN_FINAL = False`).

In [4]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.optimize import minimize
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset

PROJECT_DIR = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "exploration/features_selection.ipynb").is_file()
)

# Reuse the authoritative preprocessing by stable cell ID; no experiment cells run here.
reference = json.loads((PROJECT_DIR / "exploration/features_selection.ipynb").read_text())
previous_dir = Path.cwd()
try:
    os.chdir(PROJECT_DIR)
    for cell_id in [
        "selection-imports", "selection-history", "selection-imputation",
        "selection-features", "selection-pipeline",
    ]:
        source = "".join(next(c for c in reference["cells"] if c.get("id") == cell_id)["source"])
        exec(source)
finally:
    os.chdir(previous_dir)

DATA_DIR = PROJECT_DIR / "data"
PREPROCESSING = clone(pipeline[:2])
SEEDS = (42, 43, 44)
SPLIT_SEED = 42
CALIBRATION_FOLD = 1
VALIDATION_FOLDS = (2, 3, 4, 5)
EPOCHS, BATCH_SIZE = 160, 128
LEARNING_RATE = 0.004528032053597474
WEIGHT_DECAY = 1.2983658687387707e-5
SECOND_MODEL_SEED_OFFSET = 10_000
CACHE_DIR = DATA_DIR / "two_model_posthoc_quadratic_oof_v2"
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else "cpu"
)
torch.set_num_threads(3)
print(f"Device: {DEVICE}; calibration fold: {CALIBRATION_FOLD}; seeds: {SEEDS}")

Device: mps; calibration fold: 1; seeds: (42, 43, 44)


## Model definitions

Both Transformers have the same architecture and ordinary MSE training objective, but their weights and minibatch randomization are independent. For each base seed, the second pipeline uses `base_seed + SECOND_MODEL_SEED_OFFSET`.

After the second Transformer predicts a held-out patient, a degree-2 polynomial

\[
q(u)=a+bu+cu^2
\]

is fitted by least squares to that patient's predicted visits, where visit time `u` is scaled to `[0, 1]`. This post-processing uses no target and has no gradient connection to either Transformer.

In [5]:
def encoder_features(imputed):
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def sequence_dataset(raw, features, targets=None):
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age", "Index"])
    row_groups = [g.index.to_numpy() for _, g in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(
        np.zeros(len(raw)) if targets is None else np.asarray(targets),
        dtype=torch.float32,
    )
    return TensorDataset(
        pad_sequence([values[rows] for rows in row_groups], batch_first=True),
        pad_sequence([labels[rows] for rows in row_groups], batch_first=True),
        pad_sequence([torch.tensor(rows) for rows in row_groups], batch_first=True, padding_value=-1),
    )


class VisitTransformer(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        width, dropout = 32, 0.20773426359864192
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=8, dim_feedforward=3 * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=5, enable_nested_tensor=False)
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        self.head = nn.Sequential(
            nn.Linear(width, 16), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(16, 16), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(16, 1),
        )

    def forward(self, values, padding):
        hidden = self.encoder(
            self.projection(values), src_key_padding_mask=padding, is_causal=False
        )
        return self.head(hidden).squeeze(-1)


def fit_transformer(raw, targets, preprocessing, seed):
    torch.manual_seed(seed)
    features = encoder_features(preprocessing.transform(raw))
    scaler = StandardScaler().fit(features)
    target_mean = targets.mean()
    target_scale = max(targets.std(ddof=0), 1e-8)
    training = sequence_dataset(
        raw, scaler.transform(features), (targets - target_mean) / target_scale
    )
    network = VisitTransformer(features.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(
        network.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=1, eta_min=LEARNING_RATE * 0.01
    )
    loader = DataLoader(
        training, batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    for epoch in range(1, EPOCHS + 1):
        network.train()
        for values, labels, rows in loader:
            values, labels, rows = (x.to(DEVICE) for x in (values, labels, rows))
            valid = rows >= 0
            optimizer.zero_grad(set_to_none=True)
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], labels[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        if epoch % 40 == 0:
            print(f"seed={seed}, epoch={epoch}/{EPOCHS}", flush=True)
    network.requires_grad_(False).eval()
    return dict(
        network=network, preprocessing=preprocessing, scaler=scaler,
        target_mean=target_mean, target_scale=target_scale,
    )


@torch.no_grad()
def predict_transformer(fitted, raw):
    features = encoder_features(fitted["preprocessing"].transform(raw))
    data = sequence_dataset(raw, fitted["scaler"].transform(features))
    prediction = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(data, batch_size=BATCH_SIZE):
        values, rows = values.to(DEVICE), rows.to(DEVICE)
        valid = rows >= 0
        output = fitted["network"](values, ~valid)
        prediction[rows[valid].cpu().numpy()] = output[valid].cpu().numpy()
    return prediction * fitted["target_scale"] + fitted["target_mean"]


def project_quadratic(raw, prediction):
    # Pure post-processing: no target values and no PyTorch computation graph.
    frame = raw[["patient_id", "age", "Index"]].copy()
    frame["prediction"] = np.asarray(prediction)
    projected = pd.Series(index=raw.index, dtype=float)
    for _, visits in frame.sort_values(["patient_id", "age", "Index"]).groupby("patient_id"):
        age = visits["age"].to_numpy()
        u = (age - age[0]) / max(age[-1] - age[0], 1e-8)
        coefficients = np.polyfit(u, visits["prediction"].to_numpy(), 2)
        projected.loc[visits.index] = np.polyval(coefficients, u)
    return projected.loc[raw.index].to_numpy()

## Out-of-fold predictions

For every `(base seed, fold)`, two separate Transformers are trained. The second uses an offset seed to guarantee independent initialization and minibatch order. Its raw predictions are then quadratically post-processed outside PyTorch.

Each result is cached, so interrupted runs resume without retraining completed models. Set `RUN_CV = True` to run the complete evaluation.

In [6]:
RUN_CV = True

splits = list(GroupKFold(
    n_splits=5, shuffle=True, random_state=SPLIT_SEED
).split(X, y, groups=X["patient_id"]))
COMPONENTS = ["raw", "quadratic"]


def fit_blend_weights(frame):
    design = frame[COMPONENTS].to_numpy()
    target = frame["target"].to_numpy()
    objective = lambda weight: np.mean((target - design @ weight) ** 2)
    result = minimize(
        objective, np.full(len(COMPONENTS), 1 / len(COMPONENTS)),
        bounds=[(0.0, 1.0)] * len(COMPONENTS),
        constraints={"type": "eq", "fun": lambda weight: weight.sum() - 1.0},
        method="SLSQP",
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result.x


def fit_fold(base_seed, fold):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = CACHE_DIR / f"seed_{base_seed}_fold_{fold}.csv"
    if path.is_file():
        print(f"Loading {path.name}")
        return pd.read_csv(path)

    train_index, valid_index = splits[fold - 1]
    raw_train, raw_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
    preprocessing = clone(PREPROCESSING).fit(raw_train, y_train)
    quadratic_seed = base_seed + SECOND_MODEL_SEED_OFFSET
    with threadpool_limits(limits=3):
        raw_model = fit_transformer(raw_train, y_train, preprocessing, base_seed)
        quadratic_source_model = fit_transformer(raw_train, y_train, preprocessing, quadratic_seed)
        raw_prediction = predict_transformer(raw_model, raw_valid)
        quadratic_source = predict_transformer(quadratic_source_model, raw_valid)
        quadratic_prediction = project_quadratic(raw_valid, quadratic_source)

    result = raw_valid[["Index", "patient_id", "age"]].assign(
        row_position=valid_index, fold=fold, seed=base_seed,
        quadratic_seed=quadratic_seed, target=y_valid.to_numpy(),
        raw=raw_prediction, quadratic_source_raw=quadratic_source,
        quadratic=quadratic_prediction,
    )
    result.to_csv(path, index=False)
    print(f"Saved {path.name}: raw MSE={mean_squared_error(y_valid, raw_prediction):.6f}")
    return result


if RUN_CV:
    # Stage 1: train fold 1 only and freeze weights from seed-averaged predictions.
    calibration_oof = pd.concat(
        [fit_fold(seed, CALIBRATION_FOLD) for seed in SEEDS], ignore_index=True
    )
    calibration_average = (
        calibration_oof.groupby(
            ["row_position", "Index", "patient_id", "age", "fold", "target"],
            as_index=False,
        )[COMPONENTS].mean()
    )
    blend_weights = fit_blend_weights(calibration_average)
    print("Frozen weights:", dict(zip(COMPONENTS, blend_weights.round(6))))

    # Stage 2: only after weights are frozen, train the four validation folds.
    validation_oof = pd.concat(
        [fit_fold(seed, fold) for seed in SEEDS for fold in VALIDATION_FOLDS],
        ignore_index=True,
    )
    oof = pd.concat([calibration_oof, validation_oof], ignore_index=True)
    assert oof.groupby(["seed", "patient_id"])["fold"].nunique().eq(1).all()
    assert len(oof) == len(X) * len(SEEDS)
else:
    print("Set RUN_CV=True to train or load all OOF predictions.")

seed=42, epoch=40/160
seed=42, epoch=80/160
seed=42, epoch=120/160
seed=42, epoch=160/160
seed=10042, epoch=40/160
seed=10042, epoch=80/160
seed=10042, epoch=120/160
seed=10042, epoch=160/160
Saved seed_42_fold_1.csv: raw MSE=6.778061
seed=43, epoch=40/160
seed=43, epoch=80/160
seed=43, epoch=120/160
seed=43, epoch=160/160
seed=10043, epoch=40/160
seed=10043, epoch=80/160
seed=10043, epoch=120/160
seed=10043, epoch=160/160
Saved seed_43_fold_1.csv: raw MSE=6.894592
seed=44, epoch=40/160
seed=44, epoch=80/160
seed=44, epoch=120/160
seed=44, epoch=160/160
seed=10044, epoch=40/160
seed=10044, epoch=80/160
seed=10044, epoch=120/160
seed=10044, epoch=160/160
Saved seed_44_fold_1.csv: raw MSE=6.751292
Frozen weights: {'raw': np.float64(0.336761), 'quadratic': np.float64(0.663239)}
seed=42, epoch=40/160
seed=42, epoch=80/160
seed=42, epoch=120/160
seed=42, epoch=160/160
seed=10042, epoch=40/160
seed=10042, epoch=80/160
seed=10042, epoch=120/160
seed=10042, epoch=160/160
Saved seed_42_fold_2.c

## Calibrate once, validate once

The raw and independently post-processed pipelines are averaged over seeds before their two blend weights are fitted on fold 1. Those weights are then frozen for folds 2–5.

The paired confidence interval resamples whole patients. This confirms the frozen blend conditional on the selected model family; it is not an external test set.

In [7]:
def cluster_inference(frame, n_bootstrap=10_000, n_permutations=20_000, seed=20261108):
    scored = frame.assign(
        baseline_error=(frame["target"] - frame["raw"]) ** 2,
        candidate_error=(frame["target"] - frame["blend"]) ** 2,
    )
    clusters = scored.groupby("patient_id").agg(
        baseline_sum=("baseline_error", "sum"),
        candidate_sum=("candidate_error", "sum"),
        n=("target", "size"),
    )
    rng = np.random.default_rng(seed)
    bootstrap_gain, bootstrap_relative = [], []
    for _ in range(n_bootstrap):
        sample = clusters.iloc[rng.integers(0, len(clusters), len(clusters))]
        baseline_mse = sample["baseline_sum"].sum() / sample["n"].sum()
        candidate_mse = sample["candidate_sum"].sum() / sample["n"].sum()
        bootstrap_gain.append(baseline_mse - candidate_mse)
        bootstrap_relative.append(100 * (1 - candidate_mse / baseline_mse))

    patient_delta = (clusters["baseline_sum"] - clusters["candidate_sum"]).to_numpy()
    observed = abs(patient_delta.sum())
    exceedances = 0
    for _ in range(n_permutations):
        exceedances += abs((rng.choice([-1, 1], len(patient_delta)) * patient_delta).sum()) >= observed

    patient_mse = scored.groupby("patient_id")[["baseline_error", "candidate_error"]].mean()
    return pd.Series({
        "absolute_gain": scored["baseline_error"].mean() - scored["candidate_error"].mean(),
        "absolute_gain_CI_low": np.quantile(bootstrap_gain, 0.025),
        "absolute_gain_CI_high": np.quantile(bootstrap_gain, 0.975),
        "relative_gain_percent": 100 * (1 - scored["candidate_error"].mean() / scored["baseline_error"].mean()),
        "relative_gain_CI_low": np.quantile(bootstrap_relative, 0.025),
        "relative_gain_CI_high": np.quantile(bootstrap_relative, 0.975),
        "patient_win_rate": (patient_mse["candidate_error"] < patient_mse["baseline_error"]).mean(),
        "cluster_sign_flip_pvalue": (exceedances + 1) / (n_permutations + 1),
    })


if RUN_CV:
    seed_average = (
        oof.groupby(["row_position", "Index", "patient_id", "age", "fold", "target"], as_index=False)
        [COMPONENTS].mean()
    )
    seed_average["blend"] = seed_average[COMPONENTS].to_numpy() @ blend_weights

    validation = seed_average.loc[seed_average["fold"].isin(VALIDATION_FOLDS)].copy()
    fold_scores = validation.groupby("fold").apply(
        lambda part: pd.Series({
            "raw_MSE": mean_squared_error(part["target"], part["raw"]),
            "blend_MSE": mean_squared_error(part["target"], part["blend"]),
        }),
        include_groups=False,
    )
    fold_scores["improvement_percent"] = 100 * (1 - fold_scores["blend_MSE"] / fold_scores["raw_MSE"])

    seed_scores = []
    for seed, part in oof.loc[oof["fold"].isin(VALIDATION_FOLDS)].groupby("seed"):
        prediction = part[COMPONENTS].to_numpy() @ blend_weights
        seed_scores.append({
            "seed": seed,
            "raw_MSE": mean_squared_error(part["target"], part["raw"]),
            "blend_MSE": mean_squared_error(part["target"], prediction),
        })
    seed_scores = pd.DataFrame(seed_scores).set_index("seed")
    seed_scores["improvement_percent"] = 100 * (1 - seed_scores["blend_MSE"] / seed_scores["raw_MSE"])

    validation_summary = pd.Series({
        "raw_MSE": mean_squared_error(validation["target"], validation["raw"]),
        "blend_MSE": mean_squared_error(validation["target"], validation["blend"]),
    })
    validation_summary["improvement_percent"] = 100 * (
        1 - validation_summary["blend_MSE"] / validation_summary["raw_MSE"]
    )
    inference = cluster_inference(validation)

    print("Frozen weights:", dict(zip(COMPONENTS, blend_weights.round(6))))
    print("\nPrimary validation (folds 2–5, three-seed prediction average):")
    display(validation_summary.to_frame("value"))
    display(fold_scores)
    print("Seed sensitivity on folds 2–5:")
    display(seed_scores)
    print("Patient-cluster paired inference:")
    display(inference.to_frame("value"))

Frozen weights: {'raw': np.float64(0.336761), 'quadratic': np.float64(0.663239)}

Primary validation (folds 2–5, three-seed prediction average):


,value
raw_MSE,6.496184
blend_MSE,6.265956
improvement_percent,3.544046


,raw_MSE,blend_MSE,improvement_percent
fold,,,
2,6.489532,6.223707,4.096215
3,6.336805,6.172782,2.588426
4,6.779506,6.530316,3.675625
5,6.372113,6.131366,3.778123


Seed sensitivity on folds 2–5:


,raw_MSE,blend_MSE,improvement_percent
seed,,,
42,6.818890,6.381021,6.421414
43,6.857178,6.521939,4.888877
44,6.877857,6.413698,6.748588


Patient-cluster paired inference:


,value
absolute_gain,0.230228
absolute_gain_CI_low,0.174759
absolute_gain_CI_high,0.288578
relative_gain_percent,3.544046
relative_gain_CI_low,2.716617
relative_gain_CI_high,4.368002
patient_win_rate,0.614419
cluster_sign_flip_pvalue,0.000050


## Final fit and test prediction

This cell trains both independent Transformer pipelines on all training patients for each base seed, post-processes only the second pipeline, averages across seeds, applies the frozen two-component blend, and writes a submission. It is intentionally disabled.

In [8]:
RUN_FINAL = False

if RUN_FINAL:
    if "blend_weights" not in globals():
        raise RuntimeError("Run the OOF calibration and validation cells first.")

    X_test = pd.read_csv(DATA_DIR / "X_test_oiZ2ukx.csv")[X.columns]
    full_preprocessing = clone(PREPROCESSING).fit(X, y)
    raw_predictions, quadratic_predictions = [], []
    with threadpool_limits(limits=3):
        for base_seed in SEEDS:
            raw_model = fit_transformer(X, y, full_preprocessing, base_seed)
            quadratic_model = fit_transformer(
                X, y, full_preprocessing, base_seed + SECOND_MODEL_SEED_OFFSET
            )
            raw_predictions.append(predict_transformer(raw_model, X_test))
            quadratic_source = predict_transformer(quadratic_model, X_test)
            quadratic_predictions.append(project_quadratic(X_test, quadratic_source))

    raw_prediction = np.mean(raw_predictions, axis=0)
    quadratic_prediction = np.mean(quadratic_predictions, axis=0)
    component_predictions = np.column_stack([raw_prediction, quadratic_prediction])
    final_prediction = component_predictions @ blend_weights

    assert np.isfinite(final_prediction).all()
    submission = pd.DataFrame({"Index": X_test["Index"], "target": final_prediction})
    submission_path = DATA_DIR / "submission_raw_and_independent_quadratic.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Saved {submission_path} ({len(submission):,} visits)")